# Mobile-Money Agent Transaction Data Quality and Cleaning

This notebook audits the generated transaction extract before
cleaning, applies the reusable rules in `cleaning_pipeline.py`, and summarizes
the merged data. Unknown agents are retained for investigation rather than
discarded or assigned to an invented county.

## Load and inspect the source extracts

In [1]:
import pandas as pd
from pathlib import Path
from cleaning_pipeline import clean_agent_data

if not Path("agent_transactions.csv").is_file() or not Path("agents.csv").is_file():
    from generate_data import generate_agent_data

    generated_txns, generated_agents = generate_agent_data()
    generated_txns.to_csv("agent_transactions.csv", index=False)
    generated_agents.to_csv("agents.csv", index=False)

raw_txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
print("Transactions:", raw_txns.shape, "| Agent register:", agents.shape)
display(raw_txns.dtypes.rename("dtype").to_frame())
display(raw_txns.isna().sum().rename("missing_rows").to_frame())
print("Transaction date text examples:", raw_txns["txn_date"].head().tolist())
print("Agent IDs absent from register:", sorted(set(raw_txns["agent_id"]) - set(agents["agent_id"])))
print("Pandas treats the text 'N/A' as missing by default; inspect isna() before numeric coercion.")

Transactions: (612, 5) | Agent register: (20, 3)


,dtype
txn_id,str
agent_id,str
txn_type,str
amount,float64
txn_date,str


,missing_rows
txn_id,0
agent_id,0
txn_type,0
amount,22
txn_date,0


Transaction date text examples: ['2026-03-03', '2026-04-27', '2026-04-29', '2026-04-24', '2026-04-17']
Agent IDs absent from register: ['AG099']
Pandas treats the text 'N/A' as missing by default; inspect isna() before numeric coercion.


## Data-quality audit

In [2]:
audit_unique = raw_txns.drop_duplicates("txn_id", keep="first").copy()
normalized_type = audit_unique["txn_type"].astype("string").str.strip().str.lower()
canonical_type = normalized_type.map({"deposit": "Deposit", "withdrawal": "Withdrawal"})
numeric_amount = pd.to_numeric(audit_unique["amount"], errors="coerce")
observed_amounts = numeric_amount.dropna()
q1, q3 = observed_amounts.quantile([0.25, 0.75])
upper_fence = q3 + 1.5 * (q3 - q1)
lower_fence = q1 - 1.5 * (q3 - q1)
unknown_agents = ~audit_unique["agent_id"].isin(agents["agent_id"])

audit = pd.DataFrame(
    [
        ("Repeated transaction IDs", "Uniqueness", "txn_id", int(raw_txns["txn_id"].duplicated().sum())),
        ("Noncanonical transaction-type labels", "Consistency", "txn_type", int(audit_unique["txn_type"].astype("string").ne(canonical_type).sum())),
        ("Missing or N/A amounts", "Completeness", "amount", int(audit_unique["amount"].isna().sum())),
        ("IQR-fence amount outliers", "Accuracy", "amount", int(((numeric_amount < lower_fence) | (numeric_amount > upper_fence)).sum())),
        ("Slash-separated dates", "Validity", "txn_date", int(audit_unique["txn_date"].astype("string").str.contains("/", regex=False).sum())),
        ("Agent IDs absent from register", "Accuracy", "agent_id", int(unknown_agents.sum())),
    ],
    columns=["issue", "dimension", "column", "rows_affected"],
)
display(audit)

,issue,dimension,column,rows_affected
0,Repeated transaction IDs,Uniqueness,txn_id,12
1,Noncanonical transaction-type labels,Consistency,txn_type,376
2,Missing or N/A amounts,Completeness,amount,20
3,IQR-fence amount outliers,Accuracy,amount,40
4,Slash-separated dates,Validity,txn_date,5
5,Agent IDs absent from register,Accuracy,agent_id,7


Counts other than duplicate rows are measured on the 600 distinct
transaction IDs so a duplicated row does not inflate the issue count. The raw
extract has 612 rows; its 12 repeated IDs are separately counted above. Missing
amounts include the literal `N/A`, which `read_csv` has already parsed as
missing.

## Clean, merge and validate

In [3]:
clean_df, cleaning_log = clean_agent_data(raw_txns, agents)
Path("outputs").mkdir(exist_ok=True)
clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False,
    date_format="%Y-%m-%d",
)
print("Rows before:", len(raw_txns))
print("Rows after duplicate removal and merge:", len(clean_df))
print("Transactions with unknown agents:", int(clean_df["county"].isna().sum()))
print("Unknown-agent rows were retained with county missing.")
display(pd.DataFrame(cleaning_log))

Rows before: 612
Rows after duplicate removal and merge: 600
Transactions with unknown agents: 7
Unknown-agent rows were retained with county missing.


,step,rows_affected,details
0,deduplicate,12,Removed later rows with a repeated txn_id; kep...
1,standardize_txn_type,376,Trimmed whitespace and standardized case.
2,coerce_and_impute_amount,20,Coerced nonnumeric and negative amounts to mis...
3,invalid_amount_values,0,Nonnumeric or negative source values were trea...
4,parse_txn_date,5,Normalized slash-separated dates and parsed IS...
5,flag_iqr_outliers,53,"Flagged amounts outside IQR fences [-1540.25, ..."
6,left_merge_agent_register,7,Unknown agents retained with a missing county.


The left merge preserves all **600** unique transactions. **7**
transactions reference unknown agent IDs after deduplication; their county is
left missing and those records are excluded only from county-specific
aggregations, not from the cleaned transaction export.

## County summary by transaction type

In [4]:
known_county = clean_df.dropna(subset=["county"])
county_summary = (
    known_county.groupby(["county", "txn_type"])["amount"]
    .agg(total_value="sum", transactions="count", average_value="mean")
    .round(2)
)
display(county_summary)

total_value  transactions  average_value
county  txn_type                                            
Kiambu  Deposit        334737.0            43        7784.58
        Withdrawal      77024.5            27        2852.76
Kisumu  Deposit        436501.5            74        5898.67
        Withdrawal      76348.5            37        2063.47
Mombasa Deposit        149094.0            72        2070.75
        Withdrawal     368592.0            61        6042.49
Nairobi Deposit        543698.0           118        4607.61
        Withdrawal     161932.0            62        2611.81
Nakuru  Deposit        184944.0            60        3082.40
        Withdrawal     328568.5            39        8424.83

Nairobi has the highest deposit value (**KES 543,698**). Mombasa
has the highest withdrawal value (**KES 368,592**). The averages are computed
over the cleaned values, including explicitly flagged median imputations.

## Weekly transaction value by county

In [5]:
clean_df["week"] = clean_df["txn_date"].dt.to_period("W-SUN").astype("string")
weekly_pivot = known_county.assign(
    week=clean_df.loc[known_county.index, "week"]
).pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0,
).round(2)
display(weekly_pivot)

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267745.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28126.0,27077.0,41077.0,15341.0,29470.0,37439.0,21308.0
Mombasa,6759.0,285696.5,30504.5,25171.5,17050.0,38344.0,47119.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52887.5,43134.5,41400.5,282931.0,56813.5,56406.5,65135.0,30908.5
Nakuru,2770.5,56574.0,28730.0,39795.0,18972.5,18145.0,276609.0,26179.5,30862.0,14875.0


Weeks run Monday through Sunday. Unknown-agent transactions remain
in the cleaned file, but cannot be attributed to a county until the agent
register is corrected.

## Data Quality Report

### Accuracy
The IQR fences are approximately **KES -1,540.25 to KES 5,513.75**; **53**
transactions are flagged, including all **5** KES 250,000 records. These are
alerts, not proof of error: retain them and request receipt/ledger verification
before deciding whether any amount should be corrected. **7** unique
transactions refer to agents absent from the register; preserve them, leave
county unknown and reconcile agent IDs against the authoritative register
before county-level decisions.

### Completeness
There are **20** missing amounts in the distinct transaction set, including
source blanks and `N/A`. Coerce invalid numeric input to missing and impute
using the median for the same transaction type (overall median as fallback);
`amount_imputed` identifies each affected row. This supports aggregate
reporting without concealing which values were estimated.

### Consistency
Normalize case and surrounding whitespace so `txn_type` contains exactly
`Deposit` and `Withdrawal`; **376** unique rows required label normalization.
No category is silently discarded.

### Timeliness
Parsed transaction dates cover **1 March–29 April 2026**, consistent with the
generated two-month reporting window. The weekly summaries use Monday-to-Sunday
periods; confirm the reporting window against the production extract before
operational use.

### Validity
Convert amounts to numeric and parse dates strictly after normalizing the
observed slash separators; **5** dates needed separator normalization. The
cleaned date column is a datetime in memory and ISO `YYYY-MM-DD` in the
export. Non-numeric amounts are imputed and flagged rather than treated as
valid source values.

### Uniqueness
Remove **12** later rows with already-seen `txn_id` values, retaining the
first occurrence. This leaves **600** unique transactions from **612** source
rows without removing any distinct transaction ID.

### Privacy
If customer phone numbers were present, they would be personal data under the
Kenya Data Protection Act, 2019. Collect and use them only when necessary and
with an appropriate lawful basis; mask or tokenize them in analysis outputs,
restrict access, encrypt stored/transmitted data, and apply retention and
deletion controls. Do not include identifiable phone numbers in shared
notebooks or summary exports.